In [1]:
pip install torch torchvision pillow matplotlib scikit-learn numpy

Note: you may need to restart the kernel to use updated packages.


In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from PIL import Image
import os
import numpy as np
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

In [3]:
dataset_path = "dataset"
classes = ["spiders", "batterfly"]
image_paths = []
labels = []

for class_idx, class_name in enumerate(classes):
    class_dir = os.path.join(dataset_path, class_name)
    for img_name in os.listdir(class_dir):
        if img_name.lower().endswith(('.png', '.jpg', '.jpeg')):
            image_paths.append(os.path.join(class_dir, img_name))
            labels.append(class_idx)

print(f"Загружено изображений: {len(image_paths)}")
print(f"Классы: {classes}")

Загружено изображений: 52
Классы: ['spiders', 'batterfly']


In [4]:
# Трансформации изображений
transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])
])

class SimpleDataset(Dataset):
    def __init__(self, paths, labels, transform=None):
        self.paths = paths
        self.labels = labels
        self.transform = transform
    
    def __len__(self): 
        return len(self.paths)
    
    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert('RGB')
        if self.transform: 
            img = self.transform(img)
        return img, self.labels[idx]

train_paths, val_paths, train_labels, val_labels = train_test_split(
    image_paths, labels, test_size=0.2, random_state=42
)

train_dataset = SimpleDataset(train_paths, train_labels, transform)
val_dataset = SimpleDataset(val_paths, val_labels, transform)

batch_size = 8
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

print(f"Train: {len(train_dataset)} images, Val: {len(val_dataset)} images")

Train: 41 images, Val: 11 images


In [5]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Flatten(),
            nn.Linear(64 * 16 * 16, 128), nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(128, 2)
        )
    
    def forward(self, x):
        return self.net(x)

model = SimpleCNN()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

print(f"Параметры модели: {sum(p.numel() for p in model.parameters()):,}")

Параметры модели: 2,121,122


In [6]:
def train_epoch(model, loader):
    model.train()
    total_loss = 0
    for imgs, labels in loader:
        imgs, labels = imgs, labels
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

def evaluate(model, loader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs, labels
            outputs = model(imgs)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    return 100 * correct / total

epochs = 30
print("Начало обучения...")
for epoch in range(epochs):
    train_loss = train_epoch(model, train_loader)
    val_acc = evaluate(model, val_loader)
    print(f"Epoch {epoch+1}/{epochs}, Loss: {train_loss:.4f}, Val Acc: {val_acc:.2f}%")

Начало обучения...
Epoch 1/30, Loss: 0.7389, Val Acc: 54.55%
Epoch 2/30, Loss: 0.6683, Val Acc: 81.82%
Epoch 3/30, Loss: 0.6021, Val Acc: 81.82%
Epoch 4/30, Loss: 0.5265, Val Acc: 90.91%
Epoch 5/30, Loss: 0.3415, Val Acc: 63.64%
Epoch 6/30, Loss: 0.5991, Val Acc: 81.82%
Epoch 7/30, Loss: 0.3950, Val Acc: 81.82%
Epoch 8/30, Loss: 0.2581, Val Acc: 81.82%
Epoch 9/30, Loss: 0.1909, Val Acc: 100.00%
Epoch 10/30, Loss: 0.1501, Val Acc: 90.91%
Epoch 11/30, Loss: 0.0888, Val Acc: 100.00%
Epoch 12/30, Loss: 0.2399, Val Acc: 81.82%
Epoch 13/30, Loss: 0.1406, Val Acc: 81.82%
Epoch 14/30, Loss: 0.0691, Val Acc: 90.91%
Epoch 15/30, Loss: 0.0755, Val Acc: 72.73%
Epoch 16/30, Loss: 0.0425, Val Acc: 72.73%
Epoch 17/30, Loss: 0.0154, Val Acc: 81.82%
Epoch 18/30, Loss: 0.0073, Val Acc: 100.00%
Epoch 19/30, Loss: 0.0065, Val Acc: 100.00%
Epoch 20/30, Loss: 0.0059, Val Acc: 90.91%
Epoch 21/30, Loss: 0.0017, Val Acc: 90.91%
Epoch 22/30, Loss: 0.0031, Val Acc: 81.82%
Epoch 23/30, Loss: 0.0009, Val Acc: 81.8

In [8]:
test_path = "test_dataset"
test_paths = []
test_labels = []

for class_idx, class_name in enumerate(classes):
    class_dir = os.path.join(test_path, class_name)
    for img_name in os.listdir(class_dir):
        if img_name.lower().endswith(('.png', '.jpg', '.jpeg')):
            test_paths.append(os.path.join(class_dir, img_name))
            test_labels.append(class_idx)

test_dataset = SimpleDataset(test_paths, test_labels, transform)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

test_acc = evaluate(model, test_loader)
print(f"\nТочность на тестовых данных: {test_acc:.2f}%")


Точность на тестовых данных: 80.00%
